In [1]:
import dotenv

import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterstats

from rasterstats import zonal_stats
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask

from food_security import salinity_correction, water_quality
from food_security.fao_api import FAOClient

from pathlib import Path

In [2]:
config = dotenv.dotenv_values(".env")
username = config["FAOSTAT_USERNAME"]
password = config["FAOSTAT_PASSWORD"]

fao_client = FAOClient(username=username, password=password)

In [3]:
src_dir = Path('~').expanduser() / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/"

In [4]:
toml_file = src_dir.parent / "salinity_correction_egypt.toml"
corrected_df = salinity_correction.generate_crop_yield_csv(
    config_path=toml_file,
    add_labor=True,
    save=False,
    fao_client=fao_client
)

2026-08-05 11:07:13,142 | WARNING  | root | config input salinity_correction.crop_production.path contains a non-existing path
2026-08-05 11:07:13,143 | WARNING  | root | config input salinity_correction.mapping.path contains a non-existing path
2026-08-05 11:07:13,782 | INFO     | food_security.salinity_correction | Starting crop yield correction for Egypt
2026-08-05 11:07:13,783 | INFO     | food_security.salinity_correction | Loaded input data. Areas=68, Crops=31, Years=2


Areas:   0%|          | 0/68 [00:00<?, ?it/s]

2026-08-05 11:07:36,427 | INFO     | food_security.salinity_correction | Created dataframe with 3226 rows
2026-08-05 11:07:36,427 | INFO     | food_security.salinity_correction | Aggregating results to department level
2026-08-05 11:07:39,104 | INFO     | food_security.salinity_correction | Crop yield correction finished successfully
/Users/hemert/projects/egypt-survey-ml/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver ESRI Shapefile does not support open option CRS
  return ogr_read(


In [5]:
relevant_crops = [
    "SummerMaize_MAIZES1 (ha)",
    "NiliMaize_MAIZEN1 (ha)",
    "Wheat_WHEAT1 (ha)",
    "LongRice_PADDY1 (ha)",
    "SugarCane_SCANE1 (ha)",
    "LongBerseem_LBSEEM1 (ha)",
    "ShortBerseem_SBSEMW1 (ha)",
]

excel_salinity_df = pd.DataFrame(
    {
        "crop_name": corrected_df["crop_name"],
        "crop_name_fao": corrected_df["crop_name_fao"],
        "area": corrected_df["area_map_name"],
        "yield": corrected_df["corrected_yield"],
        "cultivation_area (ha)": corrected_df["hectares"],
        "year": corrected_df["year"],
    }
)
    
excel_salinity_df = excel_salinity_df[excel_salinity_df['crop_name'].isin(relevant_crops)]

In [6]:
excel_path = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx"

def write_excel_file(df, excel_path, sheet_name, append=False):
    # Open Excel file
    try:
        # book = load_workbook(excel_path)
        with pd.ExcelWriter(
            excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
        ) as writer:
            # excel_file.book = book
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
        print(e)
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)

In [7]:
command_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
conversion_df = pd.read_excel(excel_path, sheet_name='command_area')

mapping = (
    command_gdf.merge(
        conversion_df[['area_map_name', 'area_name']],
        left_on='OBJECTID',
        right_on='area_map_name'
    )
    .set_index('Name')['area_name']
)

excel_salinity_df['area'] = excel_salinity_df['area'].map(mapping)

In [8]:
write_excel_file(excel_salinity_df, excel_path, sheet_name="production_salinity")

In [9]:
pp_df = pd.read_excel(excel_path, sheet_name='farm_gate_price')

excel_crop_df = pd.DataFrame(
    {
        "area": excel_salinity_df["area"].unique()
    }
)

for i, row in excel_crop_df.iterrows():
    pp_total = 0
    area_name = row["area"]
    salinity_crops_row = excel_salinity_df[(excel_salinity_df['area'] == area_name) & (excel_salinity_df['year'] == 2021)]
    for crop_name in relevant_crops:
        salinity_crop_row = salinity_crops_row[salinity_crops_row['crop_name'] == crop_name]
        crop_name_fao = salinity_crop_row['crop_name_fao'].iloc[0]
        excel_crop_df.loc[i, f"salinity_production_{crop_name_fao}"] = salinity_crop_row['yield'].iloc[0]
        excel_crop_df.loc[i, f"salinity_cultivation_area_{crop_name_fao}"] = salinity_crop_row['cultivation_area (ha)'].iloc[0]

        pp_row = pp_df[pp_df['crop_name_fao'] == crop_name_fao]
        pp_crop = salinity_crop_row['yield'].iloc[0] * pp_row['Farmgate price\n(000 EGP/ton)'].iloc[0]
        pp_total += pp_crop
    
    excel_crop_df.loc[i, 'producer_price'] = pp_total

In [10]:
excel_command_df = pd.read_excel(excel_path, sheet_name='command_unit')

excel_command_df = (
    excel_command_df
    .drop(columns=excel_crop_df.columns.difference(["area"]), errors="ignore")
    .merge(excel_crop_df, on="area", how="left")
)

for column in excel_crop_df.columns:
    if column != "area" and column != 'producer_price' and 'cultivation' not in column:
        excel_command_df[column] = excel_command_df[column] / excel_command_df['rural_population']
    if column == "producer_price":
        excel_command_df[column] = excel_command_df[column] / excel_command_df['arable_km2']

write_excel_file(excel_command_df, excel_path, sheet_name='command_unit')

In [11]:
excel_command_df

,area,population,rural_population,gdp,gdp_pc,rwi_mean,rwi_median,rwi_count,water_productivity,surface_water_mean,...,salinity_cultivation_area_Maize (incl. 0067 and 0068),salinity_production_Wheat,salinity_cultivation_area_Wheat,"salinity_production_Rice, paddy","salinity_cultivation_area_Rice, paddy",salinity_production_Sugar cane,salinity_cultivation_area_Sugar cane,salinity_production_Alfalfa for forage,salinity_cultivation_area_Alfalfa for forage,producer_price
0,Blk_Air_1,4.886681e+05,17108.839844,1.811264e+08,10586.715591,0.197545,0.2365,88,3.835063e-06,0.000838,...,0.000000,1415.789044,3460.358398,0.0,0.000000,0.0,103.891922,70.708170,47.076027,1.552409e+07
1,Blk_Air_2,2.763242e+06,45841.378906,5.287341e+08,11533.990996,0.216681,0.2070,298,2.520875e-06,0.000474,...,9.747260,505.673314,3311.538086,0.0,763.791931,0.0,377.734283,56.554129,117.582001,4.934912e+06
2,Blk_Air_6,4.915848e+05,4863.967773,3.699759e+07,7606.462231,0.007000,0.0195,44,9.026268e-06,0.002594,...,24.514328,803.436450,558.269897,0.0,0.027846,0.0,505.669403,11.430953,2.116164,3.492734e+06
3,Blk_Air_7,7.106464e+06,34764.484375,3.965394e+08,11406.450585,0.283193,0.3100,419,1.478422e-05,0.003311,...,216.254822,910.378007,4521.260742,0.0,92.270279,0.0,5008.954590,3.140291,4.013838,3.382877e+06
4,Blk_Air_8,2.160475e+06,11011.501953,1.205868e+08,10950.987090,0.198614,0.2250,127,7.502828e-06,0.003443,...,29.266201,395.046154,621.435974,0.0,0.000000,0.0,705.135132,0.883233,0.403613,1.306345e+06
5,Blk_Air_9,2.190786e+06,16442.775391,1.558876e+08,9480.611721,0.427447,0.4110,159,5.701593e-05,0.003075,...,296.171906,4043.187505,9497.317383,0.0,2.038150,0.0,1119.076172,228.684538,143.115738,2.980832e+07
6,Blk_Air_11,1.388451e+06,12873.267578,7.983651e+07,6201.728334,0.112627,0.1860,75,2.508009e-05,0.000936,...,231.128860,2646.207406,4866.477539,0.0,0.000000,0.0,5632.650391,0.000000,0.000000,1.298948e+07
7,Blk_Air_12,1.473297e+06,29670.306641,2.880170e+08,9707.248456,0.154353,0.1785,156,9.939412e-08,0.001016,...,0.684528,38.839997,164.627838,0.0,88.655159,0.0,0.002532,0.218892,0.226388,6.179742e+05
8,Blk_Air_13,5.196257e+05,2629.852295,5.235418e+07,19907.650637,0.259649,0.2820,37,9.773708e-06,0.001884,...,1.831857,10005.426560,3758.970703,0.0,0.000000,0.0,235.088303,708.242818,71.442413,4.345581e+07
9,Blk_Air_14,6.171839e+05,937.707642,1.483599e+07,15821.547178,0.475200,0.5930,15,8.253039e-05,0.002611,...,13.760083,34855.219847,4669.144531,0.0,2473.478027,0.0,0.001228,98.162958,3.208595,1.858915e+08
